# T2 – Predicción de riesgo crediticio
**Curso:** Sistemas Inteligentes y Machine Learning

Este notebook implementa y compara **modelos supervisados** (Regresión Logística, Random Forest, SVM), un **modelo no supervisado** (K-Means + PCA) y una **red neuronal básica** (MLP con Keras) sobre el dataset *German Credit* (1000 solicitantes, 20 variables).

**Convención:** la clase positiva (`1`) es **mal pagador**, porque es la que más interesa detectar.

Los gráficos se guardan en la carpeta `figuras/` para insertarlos en el informe HTML.

## 0. Librerías y configuración

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

from sklearn.base import clone
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, precision_recall_curve,
                             silhouette_score)

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
os.makedirs("figuras", exist_ok=True)

def guardar(nombre):
    plt.savefig(f"figuras/{nombre}.png", dpi=150, bbox_inches="tight")

## 1. Carga de datos
Se descarga *German Credit* (`credit-g`) desde OpenML. Requiere conexión a internet (funciona en Google Colab).

In [ ]:
datos = fetch_openml("credit-g", version=1, as_frame=True)
X = datos.data.copy()
y = (datos.target == "bad").astype(int)   # 1 = mal pagador, 0 = buen pagador

print("Dimensiones:", X.shape)
print("Valores nulos totales:", int(X.isna().sum().sum()))
print("Duplicados:", int(X.duplicated().sum()))
print(y.value_counts().rename({0: "buen pagador", 1: "mal pagador"}))
X.head()

## 2. Análisis exploratorio (EDA)

In [ ]:
num_cols = X.select_dtypes("number").columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

# 2.1 Distribución de la variable objetivo
fig, ax = plt.subplots(figsize=(4, 3))
y.map({0: "Buen pagador", 1: "Mal pagador"}).value_counts().plot(kind="bar", ax=ax, color=["#1d4e89", "#c0392b"])
ax.set_title("Distribución de la clase"); ax.set_ylabel("Solicitantes"); plt.xticks(rotation=0)
guardar("eda_clase"); plt.show()

In [ ]:
# 2.2 Variables numéricas según la clase
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["duration", "credit_amount", "age"]):
    sns.boxplot(x=y.map({0: "Buen", 1: "Mal"}), y=X[col], ax=ax)
    ax.set_title(f"{col} según clase"); ax.set_xlabel("")
plt.tight_layout(); guardar("eda_numericas"); plt.show()

In [ ]:
# 2.3 Correlaciones numéricas (incluye la clase)
corr = X[num_cols].assign(mora=y).corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlaciones"); guardar("eda_correlacion"); plt.show()

In [ ]:
# 2.4 Tasa de mora por categoría de las variables más relevantes
for col in ["checking_status", "credit_history", "savings_status", "purpose"]:
    tasa = y.groupby(X[col]).mean().sort_values(ascending=False)
    plt.figure(figsize=(8, 3))
    tasa.plot(kind="bar", color="#1d4e89")
    plt.title(f"Tasa de mora por {col}"); plt.ylabel("Tasa de mora")
    plt.xticks(rotation=30, ha="right"); plt.tight_layout()
    guardar(f"eda_mora_{col}"); plt.show()

## 3. Preprocesamiento y partición
Partición **estratificada 80/20** con semilla fija. El preprocesamiento va dentro de un `Pipeline` para que se ajuste solo con datos de entrenamiento (evita fuga de datos).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("% mora train:", round(y_train.mean(), 3), "| % mora test:", round(y_test.mean(), 3))

## 4. Funciones de evaluación
Se usa un **costo asimétrico** (según la documentación del dataset): un falso negativo (aprobar a un mal pagador) cuesta **5** y un falso positivo (rechazar a un buen pagador) cuesta **1**.

In [ ]:
COSTO_FN, COSTO_FP = 5, 1
resultados = []   # métricas por modelo
probs = {}        # probabilidades de la clase "mal pagador" en test

def evaluar(nombre, y_true, y_prob, umbral=0.5, mostrar_cm=True):
    y_pred = (y_prob >= umbral).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_true, y_pred).ravel()
    fila = {
        "Modelo": nombre, "Umbral": umbral,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall (mora)": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "VN": vn, "FP": fp, "FN": fn, "VP": vp,
        "Costo": COSTO_FN * fn + COSTO_FP * fp,
    }
    if mostrar_cm:
        ConfusionMatrixDisplay.from_predictions(
            y_true, y_pred, display_labels=["Buen pagador", "Mal pagador"], cmap="Blues")
        plt.title(f"Matriz de confusión – {nombre} (umbral {umbral:.2f})")
        guardar("cm_" + nombre.lower().replace(" ", "_")); plt.show()
    return fila

## 5. Modelos supervisados
Se ajustan los hiperparámetros con `GridSearchCV` (5 folds estratificados) optimizando **F1** de la clase mal pagador, y se usa `class_weight="balanced"` para el desbalance 70/30.

In [ ]:
modelos = {
    "Regresion Logistica": (
        Pipeline([("prep", prep), ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED))]),
        {"clf__C": [0.01, 0.1, 1, 10]}),
    "Random Forest": (
        Pipeline([("prep", prep), ("clf", RandomForestClassifier(class_weight="balanced", random_state=SEED))]),
        {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 6, 10], "clf__min_samples_leaf": [1, 3]}),
    "SVM": (
        Pipeline([("prep", prep), ("clf", SVC(class_weight="balanced", probability=True, random_state=SEED))]),
        {"clf__C": [0.5, 1, 5], "clf__gamma": ["scale", 0.01]}),
}

mejores = {}
cv_tabla = []
for nombre, (pipe, grid) in modelos.items():
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="f1", n_jobs=-1)
    gs.fit(X_train, y_train)
    mejores[nombre] = gs.best_estimator_
    cv_tabla.append({"Modelo": nombre,
                     "F1 CV (media)": gs.best_score_,
                     "F1 CV (desv.)": gs.cv_results_["std_test_score"][gs.best_index_],
                     "Mejores hiperparámetros": gs.best_params_})
    probs[nombre] = gs.best_estimator_.predict_proba(X_test)[:, 1]
    resultados.append(evaluar(nombre, y_test, probs[nombre]))

pd.DataFrame(cv_tabla)

## 6. Modelo no supervisado: segmentación de clientes (K-Means + PCA)
Se agrupa **sin usar la etiqueta**. La etiqueta se usa solo después para interpretar cada grupo.

In [ ]:
prep_u = clone(prep)
X_all = prep_u.fit_transform(X)

ks = range(2, 9)
inercia, silueta = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_all)
    inercia.append(km.inertia_)
    silueta.append(silhouette_score(X_all, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(list(ks), inercia, "o-"); ax[0].set_title("Método del codo"); ax[0].set_xlabel("k"); ax[0].set_ylabel("Inercia")
ax[1].plot(list(ks), silueta, "o-", color="#c0392b"); ax[1].set_title("Coeficiente de silueta"); ax[1].set_xlabel("k")
plt.tight_layout(); guardar("kmeans_codo_silueta"); plt.show()
print({k: round(s, 3) for k, s in zip(ks, silueta)})

In [ ]:
K = 3   # <-- ajusta según el codo y la silueta del gráfico anterior
km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X_all)
etiquetas = km.labels_

pca = PCA(n_components=2, random_state=SEED)
Z = pca.fit_transform(X_all)
plt.figure(figsize=(6, 5))
sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=etiquetas, palette="Set2", s=25)
plt.title(f"Clusters K-Means (k={K}) en 2D con PCA – varianza explicada: {pca.explained_variance_ratio_.sum():.1%}")
plt.xlabel("PC1"); plt.ylabel("PC2"); guardar("kmeans_pca"); plt.show()

perfil = X[["duration", "credit_amount", "age"]].assign(cluster=etiquetas, mora=y.values) \
    .groupby("cluster").agg(n=("mora", "size"), tasa_mora=("mora", "mean"),
                            duracion=("duration", "mean"), monto=("credit_amount", "mean"), edad=("age", "mean"))
display(perfil.round(2))

## 7. Red neuronal básica (MLP con Keras)
Misma partición que los demás modelos. Se usa `class_weight`, `Dropout` y *early stopping* para limitar el sobreajuste.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers

tf.random.set_seed(SEED)
prep_nn = clone(prep)
Xtr = prep_nn.fit_transform(X_train).astype("float32")
Xte = prep_nn.transform(X_test).astype("float32")

pesos = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weight = {0: pesos[0], 1: pesos[1]}

mlp = keras.Sequential([
    layers.Input(shape=(Xtr.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
mlp.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy", keras.metrics.AUC(name="auc")])
mlp.summary()

es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
hist = mlp.fit(Xtr, y_train.values, validation_split=0.2, epochs=300, batch_size=32,
               class_weight=class_weight, callbacks=[es], verbose=0)
print("Épocas entrenadas:", len(hist.history["loss"]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="validación")
ax[0].set_title("Pérdida"); ax[0].set_xlabel("Época"); ax[0].legend()
ax[1].plot(hist.history["auc"], label="train"); ax[1].plot(hist.history["val_auc"], label="validación")
ax[1].set_title("AUC"); ax[1].set_xlabel("Época"); ax[1].legend()
plt.tight_layout(); guardar("mlp_curvas"); plt.show()

probs["MLP"] = mlp.predict(Xte, verbose=0).ravel()
resultados.append(evaluar("MLP", y_test, probs["MLP"]))

## 8. Comparación de modelos

In [ ]:
tabla = pd.DataFrame(resultados).set_index("Modelo")
cols = ["Accuracy", "Precision", "Recall (mora)", "F1", "ROC-AUC", "PR-AUC", "VN", "FP", "FN", "VP", "Costo"]
display(tabla[cols].round(3))
tabla[cols].round(3).to_csv("figuras/tabla_comparacion.csv")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for nombre, p in probs.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    ax[0].plot(fpr, tpr, label=f"{nombre} (AUC={roc_auc_score(y_test, p):.2f})")
    pr, rc, _ = precision_recall_curve(y_test, p)
    ax[1].plot(rc, pr, label=f"{nombre} (AP={average_precision_score(y_test, p):.2f})")
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set_title("Curvas ROC"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].legend()
ax[1].axhline(y_test.mean(), color="k", ls="--"); ax[1].set_title("Curvas Precision-Recall"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].legend()
plt.tight_layout(); guardar("curvas_roc_pr"); plt.show()

## 9. Ajuste del umbral por costo
El umbral de 0,5 rara vez es el mejor cuando los errores cuestan distinto. Para **no usar el conjunto de prueba al elegir**, el umbral se obtiene con predicciones de validación cruzada sobre el entrenamiento y luego se aplica en test. Se hace con el mejor modelo de scikit-learn según ROC-AUC.

In [ ]:
mejor_nombre = max(mejores, key=lambda n: roc_auc_score(y_test, probs[n]))
mejor = mejores[mejor_nombre]
print("Mejor modelo scikit-learn (ROC-AUC en test):", mejor_nombre)

p_cv = cross_val_predict(clone(mejor), X_train, y_train, cv=cv, method="predict_proba")[:, 1]
umbrales = np.linspace(0.05, 0.95, 91)
costos = []
for u in umbrales:
    pred = (p_cv >= u).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_train, pred).ravel()
    costos.append(COSTO_FN * fn + COSTO_FP * fp)
u_opt = float(umbrales[int(np.argmin(costos))])

plt.figure(figsize=(6, 3.5))
plt.plot(umbrales, costos); plt.axvline(u_opt, color="r", ls="--", label=f"umbral óptimo = {u_opt:.2f}")
plt.axvline(0.5, color="gray", ls=":", label="umbral 0,5")
plt.xlabel("Umbral"); plt.ylabel("Costo en validación cruzada"); plt.legend(); plt.title("Costo vs. umbral")
guardar("umbral_costo"); plt.show()

fila_u = evaluar(f"{mejor_nombre} umbral optimo", y_test, probs[mejor_nombre], umbral=u_opt)
display(pd.DataFrame([r for r in resultados if r["Modelo"] == mejor_nombre] + [fila_u]).set_index("Modelo")[cols + ["Umbral"]].round(3))

## 10. Análisis de errores
Se analizan los errores del mejor modelo de scikit-learn con el umbral óptimo.

In [ ]:
pred = (probs[mejor_nombre] >= u_opt).astype(int)
df_err = X_test.copy()
df_err["real"] = y_test.values; df_err["pred"] = pred; df_err["prob_mora"] = probs[mejor_nombre].round(3)

fn_df = df_err[(df_err.real == 1) & (df_err.pred == 0)]
fp_df = df_err[(df_err.real == 0) & (df_err.pred == 1)]
print(f"Falsos negativos: {len(fn_df)} | Falsos positivos: {len(fp_df)}")

print("\nFalsos negativos (mal pagadores clasificados como buenos):")
display(fn_df.sort_values("prob_mora").head(8)[["duration", "credit_amount", "age", "credit_history", "purpose", "checking_status", "prob_mora"]])

print("Comparación de perfiles promedio:")
display(df_err.assign(tipo=np.select(
    [(df_err.real == 1) & (df_err.pred == 1), (df_err.real == 1) & (df_err.pred == 0),
     (df_err.real == 0) & (df_err.pred == 1), (df_err.real == 0) & (df_err.pred == 0)],
    ["VP", "FN", "FP", "VN"], default="")).groupby("tipo")[["duration", "credit_amount", "age"]].mean().round(1))

In [ ]:
# Desempeño por subgrupo de edad (revisión de sesgo)
grupos = pd.cut(df_err["age"], [0, 25, 35, 50, 120], labels=["<=25", "26-35", "36-50", ">50"])
filas = []
for g in grupos.cat.categories:
    m = grupos == g
    sub = df_err[m]
    filas.append({"grupo_edad": g, "n": int(m.sum()), "tasa_mora_real": sub.real.mean(),
                  "recall_mora": recall_score(sub.real, sub.pred) if sub.real.sum() else np.nan,
                  "tasa_rechazo": sub.pred.mean()})
display(pd.DataFrame(filas).round(3))

In [ ]:
# Importancia de variables por permutación (sobre las variables originales)
imp = permutation_importance(mejor, X_test, y_test, scoring="roc_auc", n_repeats=15, random_state=SEED, n_jobs=-1)
orden = np.argsort(imp.importances_mean)[::-1][:10]
plt.figure(figsize=(7, 4))
plt.barh([X.columns[i] for i in orden][::-1], imp.importances_mean[orden][::-1], color="#1d4e89")
plt.title(f"Importancia por permutación – {mejor_nombre}"); plt.xlabel("Caída de ROC-AUC")
plt.tight_layout(); guardar("importancia_variables"); plt.show()

## 11. Qué llevar al informe
1. Tabla de la sección 8 (`figuras/tabla_comparacion.csv`) y las matrices de confusión (`figuras/cm_*.png`).
2. Curvas ROC/PR, curvas del MLP, codo/silueta y PCA de K-Means.
3. Resultados del umbral óptimo (sección 9) y del análisis de errores y subgrupos (sección 10).
4. Redacta cada conclusión citando un número de estas tablas.

**Advertencia:** con solo 1000 registros y 200 en test, las diferencias pequeñas entre modelos pueden ser ruido. Repite con otras semillas o usa validación cruzada completa para sustentar cuál modelo es realmente mejor.